<a href="https://colab.research.google.com/github/GeorgiosMavrakis/introduction-to-github/blob/main/Copy_of_%5BMAM11A_2026%5D_V%26L_Practicum_Uncertainty_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Practicum Vision & Language NLP - Uncertainty Quantification with Multimodal LLMs

In this notebook, we will evaluate several multimodal Vision & Language LLMs on a Medical Visual Question Answering task. We will use the following dataset: [GMAI-MMBench: A Comprehensive Multimodal Evaluation Benchmark Towards General Medical AI.](https://huggingface.co/datasets/OpenGVLab/GMAI-MMBench) The dataset is released under the Apache 2.0 license, and you will work on a pre-processed subset prepared by us.

We will evaluate several models:
- `Qwen2-VL-2B-Instruct` and `Medgemma-4b-it` (smaller models that can be run directly on Colab; output files are also provided in case of issues).
- `Qwen2.5-VL-32B-Instruct` and `GPT-5-mini` (larger models - we provide the output files).

Output files are named according to the following template:
`outputs_{model_name}-sample_{sample_answers}-temp_{temperature}.tsv`

### Setup

- In Runtime, select Change runtime type, then choose **T4 GPU** as the hardware accelerator.  

- Log into Hugging Face and accept the license agreement for [Qwen2-VL-2B-Instruct](https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct) and [Medgemma-4b-it](https://huggingface.co/google/medgemma-4b-it). Attention, this may take a while to be processed.

- Set your Hugging Face token as the variable **HF_TOKEN** in the Secrets tab (left menu, key icon) or paste your token in the "your_token" in the required field. Make sure not to accidentaly share it.


### Important Note

This challenging assignment gives you plenty of freedom to explore and be creative. You are expected to demonstrate independence, formulate your own hypotheses, and test them empirically. Since this is a cutting-edge research area with many open questions, take the opportunity to investigate, experiment, and follow your curiosity! :)


In [6]:
from typing import List, Dict, Any, Tuple, Generator, Optional, Union
import random
import string, os, sys, glob
import pandas as pd
import numpy as np
from google.colab import userdata

from PIL import Image
from tqdm import tqdm
import torch
import ast

In [7]:
try:
  hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
  hf_token = "your_token"

os.environ["HF_TOKEN"] = hf_token

In [8]:
!gdown --folder https://drive.google.com/drive/folders/1dNoFV9GvDhMdTjM2P2oHW00-f9oPMxnf -O /content/data # download pre-processed dataset files + model responses
!unzip -q data/filtered_images.zip -d data/images/ # unzip the images folder

Retrieving folder contents
Retrieving folder 1MxSEpKZWI26Gl2dVdvRd2uMn7Eimr9dC model_responses
Processing file 1tG8zYy11gXQ-puRQocji6OOKogHR12Pi outputs_gpt5-mini_no-logprobs.tsv
Processing file 1GVFn2qKdDJxtEbbrufC0W7eULvAZMyPa outputs_medgemma-4b-it-temp_0.2.tsv
Processing file 1BAx2jUeuUbgRX6joWR94vdS0nsqCNf-I outputs_medgemma-4b-it-temp_0.6.tsv
Processing file 1JK7ACAkPVKIyEZ39RtVru3GDgUvMk_8J outputs_medgemma-4b-it-temp_1.0.tsv
Processing file 18cSUZl93yG167twxUY_gmEaMYXDOfi3q outputs_Qwen2-VL-2B-Instruct-temp_0.2.tsv
Processing file 1pxUlsk9rK67Uo9mOSxKKOebMzGxPYXS9 outputs_Qwen2-VL-2B-Instruct-temp_0.6.tsv
Processing file 1WlgSroLtv5STSufSC4qRM419KW5YRBod outputs_Qwen2-VL-2B-Instruct-temp_1.0.tsv
Processing file 1HjbP_JjNq1gkqsz-RiXff-c5M_tvPvb8 outputs_Qwen2.5-VL-32B-Instruct-temp_0.2.tsv
Processing file 11XjxhRI7j-xOEtnTYnE7-v8pFTOk6jIh outputs_Qwen2.5-VL-32B-Instruct-temp_0.6.tsv
Processing file 1FSfXTQj2ytMsCfO0Iy_xoeP0P-IqI-K7 outputs_Qwen2.5-VL-32B-Instruct-temp_1.0.tsv
Pr

In [ ]:
!ls data data/model_responses

pd.set_option("display.max_colwidth", None)
ds = pd.read_csv("data/GMAI_mm_bench_dataset.tsv", sep="\t")  # separate name so df is not overwritten
print(ds.columns.tolist())
print(ds.iloc[0])

for p in sorted(glob.glob("data/model_responses/*.tsv")):
    r = pd.read_csv(p, sep="\t")
    print(os.path.basename(p), r.shape, r.columns.tolist())

print(pd.read_csv(sorted(glob.glob("data/model_responses/*Qwen2-VL-2B*.tsv"))[0], sep="\t").iloc[0])

data:
filtered_images.zip  GMAI_mm_bench_dataset.tsv	images	model_responses

data/model_responses:
outputs_gpt5-mini_no-logprobs.tsv
outputs_medgemma-4b-it-temp_0.2.tsv
outputs_medgemma-4b-it-temp_0.6.tsv
outputs_medgemma-4b-it-temp_1.0.tsv
outputs_Qwen2.5-VL-32B-Instruct-temp_0.2.tsv
outputs_Qwen2.5-VL-32B-Instruct-temp_0.6.tsv
outputs_Qwen2.5-VL-32B-Instruct-temp_1.0.tsv
outputs_Qwen2-VL-2B-Instruct-temp_0.2.tsv
outputs_Qwen2-VL-2B-Instruct-temp_0.6.tsv
outputs_Qwen2-VL-2B-Instruct-temp_1.0.tsv
['index', 'question', 'A', 'B', 'C', 'D', 'E', 'answer', 'category', 'clinical VQA task', 'department', 'perceptual granularity', 'modality', 'original task', 'image_path']
index                                                                                                  398
question                  Which of the following options best matches the marked organ in the X-ray image?
A                                                                       fascia uncovered kidney parenchyma
B   

# 1. Inference (run on GPUs)

## Load and Prepare Model Inputs

In [ ]:
def add_options_dict(df_raw: pd.DataFrame) -> pd.DataFrame:  # prepare the multiple-choice question format
    option_cols = [col for col in df_raw.columns if col in list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")]

    def _make_dict(row):
        return {col: row[col] for col in option_cols if pd.notna(row[col])}

    out = df_raw.copy()
    out["options"] = out.apply(_make_dict, axis=1)
    return out

def construct_prompt(row) -> str:  # prepare the prompt for the model
    PROMPT_TEMPLATE = """Question: ({question})
Provide a concise answer and start your response with the letter of the selected option in square brackets. Options:
(
{options}
)
Your Answer:
"""
    question = row["question"]
    options = {
        letter: row[letter]
        for letter in string.ascii_uppercase[:5]
        if letter in row and row[letter] and str(row[letter]).strip() and not pd.isna(row[letter])
    }

    formatted_options = "\n".join(
        f"[{letter}] {option}"
        for letter, option in options.items()
    )

    return PROMPT_TEMPLATE.format(
        question=question,
        options=formatted_options
    )

In [ ]:
input_tsv = "data/GMAI_mm_bench_dataset.tsv"  # the pre-processed dataset we provide
df = pd.read_csv(input_tsv, sep="\t")

df = add_options_dict(df)
df["prompt"] = df.apply(lambda row: construct_prompt(row), axis=1)
data_records: List[dict] = df.to_dict(orient="records")

In [ ]:
df.head()  # inspect the data

,index,question,A,B,C,D,E,answer,category,clinical VQA task,department,perceptual granularity,modality,original task,image_path,options,prompt
0,398,Which of the following options best matches the marked organ in the X-ray image?,fascia uncovered kidney parenchyma,lung,fascia covered kidney parenchyma,heart,clavicles,D,heart,Organ Recognition - Thorax,Cardiovascular Surgery,Contour level,X-ray,SEG_2D,398.jpg,"{'A': 'fascia uncovered kidney parenchyma', 'B': 'lung', 'C': 'fascia covered kidney parenchyma', 'D': 'heart', 'E': 'clavicles'}",Question: (Which of the following options best matches the marked organ in the X-ray image?)\nProvide a concise answer and start your response with the letter of the selected option in square brackets. Options:\n(\n[A] fascia uncovered kidney parenchyma\n[B] lung\n[C] fascia covered kidney parenchyma\n[D] heart\n[E] clavicles\n)\nYour Answer:\n
1,4322,"Observe the image, please select the option that best describes the severity depicted in the picture.",grade 2 (minimal),grade 3 (moderate),grade 4 (severe),grade 1 (doubtful),NaN,A,grade 2 (minimal),Severity Grading,Sports Medicine,Image level,X-ray,CLS_2D,4322.jpg,"{'A': 'grade 2 (minimal)', 'B': 'grade 3 (moderate)', 'C': 'grade 4 (severe)', 'D': 'grade 1 (doubtful)'}","Question: (Observe the image, please select the option that best describes the severity depicted in the picture.)\nProvide a concise answer and start your response with the letter of the selected option in square brackets. Options:\n(\n[A] grade 2 (minimal)\n[B] grade 3 (moderate)\n[C] grade 4 (severe)\n[D] grade 1 (doubtful)\n)\nYour Answer:\n"
2,2627,"Considering the symptoms visible in the picture, which option corresponds most accurately?",COVID 19,viral pneumonia,normal chest x ray,NaN,NaN,B,viral pneumonia,Disease Diagnosis,Pulmonary Medicine,Image level,X-ray,CLS_2D,2627.jpg,"{'A': 'COVID 19', 'B': 'viral pneumonia', 'C': 'normal chest x ray'}","Question: (Considering the symptoms visible in the picture, which option corresponds most accurately?)\nProvide a concise answer and start your response with the letter of the selected option in square brackets. Options:\n(\n[A] COVID 19\n[B] viral pneumonia\n[C] normal chest x ray\n)\nYour Answer:\n"
3,70,Focus on the square-highlighted area of this X-ray image. What could be the potential diagnosis?,pulmonary mass,diffuse pulmonary nodule,pleural effusion,pulmonary calcification,NaN,A,pulmonary mass,Disease Diagnosis,Pulmonary Medicine,Box level,X-ray,DET_2D,70.jpg,"{'A': 'pulmonary mass', 'B': 'diffuse pulmonary nodule', 'C': 'pleural effusion', 'D': 'pulmonary calcification'}",Question: (Focus on the square-highlighted area of this X-ray image. What could be the potential diagnosis?)\nProvide a concise answer and start your response with the letter of the selected option in square brackets. Options:\n(\n[A] pulmonary mass\n[B] diffuse pulmonary nodule\n[C] pleural effusion\n[D] pulmonary calcification\n)\nYour Answer:\n
4,4525,Choose the description that best matches the image shown.,negative for pneumonia,typical appearance of COVID-19,indeterminate appearance of COVID-19,atypical appearance of COVID-19,NaN,C,indeterminate appearance of COVID-19,Severity Grading,Pulmonary Medicine,Image level,X-ray,CLS_2D,4525.jpg,"{'A': 'negative for pneumonia', 'B': 'typical appearance of COVID-19', 'C': 'indeterminate appearance of COVID-19', 'D': 'atypical appearance of COVID-19'}",Question: (Choose the description that best matches the image shown.)\nProvide a concise answer and start your response with the letter of the selected option in square brackets. Options:\n(\n[A] negative for pneumonia\n[B] typical appearance of COVID-19\n[C] indeterminate appearance of COVID-19\n[D] atypical appearance of COVID-19\n)\nYour Answer:\n


In [ ]:
len(df) # we have 300 image-question pairs

300

In [ ]:
from transformers import AutoTokenizer

ds = pd.read_csv("data/GMAI_mm_bench_dataset.tsv", sep="\t")  # fresh copy, df is overwritten later in the notebook

# q1
for col in ["clinical VQA task", "department"]:
    vc = ds[col].value_counts()
    print(pd.DataFrame({"n": vc, "%": (100 * vc / len(ds)).round(1)}).to_string(), "\n")

# q2
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
lengths = pd.DataFrame({
    "words": ds["question"].str.split().str.len(),
    "qwen2-vl tokens": ds["question"].apply(lambda q: len(tok(q, add_special_tokens=False)["input_ids"])),
})
print(lengths.describe().round(2).to_string(), "\n")

ends_q = ds["question"].str.strip().str.endswith("?")
print(f"ending with '?': {ends_q.sum()}/{len(ds)} ({100 * ends_q.mean():.1f}%)\n")

opts = ds[list("ABCDE")]
n_opts = (opts.notna() & opts.astype(str).apply(lambda c: c.str.strip() != "")).sum(axis=1)  # same rule as construct_prompt
print(n_opts.value_counts().sort_index().rename("questions").to_string(), "\n")
gold = ds["answer"].value_counts().reindex(list("ABCDE"), fill_value=0)
print(gold.rename("questions").to_string(), "\n")

modal = gold[gold == gold.max()].index.tolist()
print(f"(a) uniform random baseline: {(1 / n_opts).mean():.4f}")
print(f"(b) always-'{modal[0]}' baseline: {(ds['answer'] == modal[0]).mean():.4f}  (modal letters: {modal})")

                                     n     %
clinical VQA task                           
Disease Diagnosis                  182  60.7
Organ Recognition - Thorax          25   8.3
Bone                                22   7.3
Severity Grading                    16   5.3
Organ Recognition - Abdomen         14   4.7
Organ Recognition - Head and Neck   13   4.3
Blood Vessels Recognition           10   3.3
Muscle                               7   2.3
Organ Recognition - Pelvic           7   2.3
Attribute Recognition                4   1.3 

                                             n     %
department                                          
Pulmonary Medicine                          76  25.3
Sports Medicine                             50  16.7
Orthopedic Surgery                          49  16.3
Gastroenterology and Hepatology             32  10.7
Oncology (Medical)                          17   5.7
Hematology                                  14   4.7
Cardiovascular Surgery            

## a. Inference with Qwen2-VL-2B-Instruct

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor, GenerationConfig
from PIL import Image

model_id = "Qwen/Qwen2-VL-2B-Instruct"

processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype='auto',
    device_map='auto'
)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype='auto',
    device_map='auto'
).eval()

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

In [ ]:
model.device # verify this is running on 'cuda'

device(type='cuda', index=0)

In [ ]:
images_dir = "data/images"

def format_inputs(batch_item):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": batch_item["prompt"]},
                {"type": "image"},
            ],
        }
    ]

    return {
        **batch_item,
        "messages": messages,
        "image": Image.open(os.path.join(images_dir, batch_item["image_path"]))
    }

In [ ]:
from transformers import TemperatureLogitsWarper, TopKLogitsWarper, TopPLogitsWarper

print({k: getattr(model.generation_config, k) for k in ["do_sample", "temperature", "top_p", "top_k"]})

qid = 398
item = format_inputs(next(r for r in data_records if r["index"] == qid))
prompt = processor.apply_chat_template(item["messages"], add_generation_prompt=True)
inputs = processor(images=[item["image"]], text=[prompt], return_tensors="pt", padding=True).to(model.device)

torch.manual_seed(0)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=32, do_sample=True, temperature=0.6, top_p=0.9,
                             return_dict_in_generate=True, output_scores=True, output_logits=True,
                             pad_token_id=processor.tokenizer.pad_token_id)

# q4: the two notebook lines on one live response
b_idx = 0
input_len = inputs["input_ids"][b_idx].shape[0]
generated_tokens = outputs.sequences[b_idx, input_len:]
print("\ntoken ids:", generated_tokens.tolist())
print("decode, skip_special_tokens=True: ", repr(processor.decode(generated_tokens, skip_special_tokens=True)))
print("decode, skip_special_tokens=False:", repr(processor.decode(generated_tokens, skip_special_tokens=False)))

probs = torch.nn.functional.softmax(outputs.logits[0][b_idx].float(), dim=-1)  # step 0, raw logits
print(f"\nsoftmax at step 0: shape {tuple(probs.shape)}, sum {probs.sum().item():.6f}")
top5 = probs.topk(5)
for p, i in zip(top5.values.tolist(), top5.indices.tolist()):
    print(f"  {processor.tokenizer.decode([i])!r:>12}  p={p:.4f}")
tid0 = generated_tokens[0].item()
p_used = torch.softmax(outputs.scores[0][b_idx].float(), dim=-1)[tid0].item()  # processed scores actually sampled from
print(f"sampled token {processor.tokenizer.decode([tid0])!r}: p(raw logits)={probs[tid0].item():.4f}, p(processed scores)={p_used:.4f}")

pairs = []
for j, logits in enumerate(outputs.logits):
    t_id = outputs.sequences[b_idx, input_len + j].item()
    lp = torch.log(torch.nn.functional.softmax(logits[b_idx], dim=-1)[t_id]).item()
    pairs.append((processor.tokenizer.decode([t_id]), lp))
print("\ntoken-logprob pairs:")
for t, lp in pairs:
    print(f"  {t!r:>16}  logprob={lp:.4f}  p={np.exp(lp):.4f}")

# q3: step-0 distribution after temperature and top-p, with and without the inherited top_k=1
raw = outputs.logits[0][b_idx].float()[None]
rows = []
for T in [0.2, 0.6, 1.0, 1.5]:
    for name, warpers in [("top_p=0.9", [TemperatureLogitsWarper(T), TopPLogitsWarper(0.9)]),
                          ("top_k=1, top_p=0.9", [TemperatureLogitsWarper(T), TopKLogitsWarper(1), TopPLogitsWarper(0.9)])]:
        s = raw
        for w in warpers:
            s = w(None, s)
        p = torch.softmax(s, dim=-1)[0]
        for i in torch.isfinite(s[0]).nonzero().flatten().tolist():
            rows.append({"filter": name, "temperature": T, "token": processor.tokenizer.decode([i]), "prob": p[i].item()})

q3 = pd.DataFrame(rows)
q3["letter"] = q3["token"].str.extract(r"^\s*\[?([A-E])\]?\s*$", expand=False).fillna("other")
piv = q3.pivot_table(index=["filter", "temperature"], columns="letter", values="prob", aggfunc="sum", fill_value=0)
piv["n_tokens"] = q3.groupby(["filter", "temperature"]).size()
print(f"\nquestion {qid}: {item['question']}\noptions: {item['options']}\ngold: {item['answer']}\n")
print(piv.round(4).to_string())

os.makedirs("outputs", exist_ok=True)
q3.sort_values(["filter", "temperature", "prob"], ascending=[True, True, False]).to_csv(
    "outputs/q3_qwen2-vl-2b_idx398_first_token_temp_topp.csv", index=False)
pd.DataFrame([{"index": qid, "prompt": prompt, "response": processor.decode(generated_tokens, skip_special_tokens=True),
               "token_ids": generated_tokens.tolist(), "logprobs": pairs}]).to_csv(
    "outputs/q4_qwen2-vl-2b_idx398_live_response_logprobs.tsv", sep="\t", index=False)

{'do_sample': True, 'temperature': 0.01, 'top_p': 0.001, 'top_k': 1}

token ids: [32622, 60, 20622, 151645]
decode, skip_special_tokens=True:  '[B] lung'
decode, skip_special_tokens=False: '[B] lung<|im_end|>'

softmax at step 0: shape (151936,), sum 1.000001
          '[B'  p=0.5148
          '[C'  p=0.1894
          '[A'  p=0.1014
          '[D'  p=0.0697
           '['  p=0.0290
sampled token '[B': p(raw logits)=0.5148, p(processed scores)=1.0000

token-logprob pairs:
              '[B'  logprob=-0.6640  p=0.5148
               ']'  logprob=-0.0009  p=0.9991
           ' lung'  logprob=-0.0070  p=0.9930
      '<|im_end|>'  logprob=-0.0001  p=0.9999

question 398: Which of the following options best matches the marked organ in the X-ray image?
options: {'A': 'fascia uncovered kidney parenchyma', 'B': 'lung', 'C': 'fascia covered kidney parenchyma', 'D': 'heart', 'E': 'clavicles'}
gold: D

letter                               A       B       C       D       E   other  n_tokens
filter 

In [ ]:
qid = 398
for t in ["0.2", "0.6", "1.0"]:
    out = pd.read_csv(f"data/model_responses/outputs_Qwen2-VL-2B-Instruct-temp_{t}.tsv", sep="\t")
    out["letter"] = out["response"].astype(str).str.extract(r"\[([A-E])\]", expand=False)  # same regex as parse_response
    n_q = out["index"].nunique()
    multi_letter = (out.groupby("index")["letter"].nunique(dropna=False) > 1).sum()
    multi_text = (out.groupby("index")["response"].nunique(dropna=False) > 1).sum()
    q = out[out["index"] == qid]
    first = sorted({(lp[0][0], round(lp[0][1], 4)) for lp in q["logprobs"].apply(ast.literal_eval)})
    print(f"T={t}: >1 distinct letter in {multi_letter}/{n_q} questions, >1 distinct response text in {multi_text}/{n_q}")
    print(f"  q{qid} letters: {q['letter'].tolist()}")
    print(f"  q{qid} first-token (token, logprob): {first}")

row = out.iloc[0:0]
row = pd.read_csv("data/model_responses/outputs_Qwen2-VL-2B-Instruct-temp_0.6.tsv", sep="\t")
row = row[row["index"] == qid].iloc[0]
print(f"\nstored T=0.6 sample 1 for q{qid}: {row['response']!r}")
for t, lp in ast.literal_eval(row["logprobs"]):
    print(f"  {t!r:>18}  logprob={lp:.4f}  p={np.exp(lp):.4f}")

T=0.2: >1 distinct letter in 84/300 questions, >1 distinct response text in 115/300
  q398 letters: ['B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B']
  q398 first-token (token, logprob): [('[B', -0.7649)]
T=0.6: >1 distinct letter in 216/300 questions, >1 distinct response text in 242/300
  q398 letters: ['B', 'B', 'B', 'C', 'B', 'B', 'C', 'B', 'C', 'B']
  q398 first-token (token, logprob): [('[B', -0.7649), ('[C', -1.5149)]
T=1.0: >1 distinct letter in 284/300 questions, >1 distinct response text in 291/300
  q398 letters: ['B', 'C', 'C', 'B', 'C', 'B', 'B', 'B', 'D', 'B']
  q398 first-token (token, logprob): [('[B', -0.7649), ('[C', -1.5149), ('[D', -2.6399)]

stored T=0.6 sample 1 for q398: '[B] lung'
                '[B'  logprob=-0.7649  p=0.4654
                 ']'  logprob=-0.0010  p=0.9990
             ' lung'  logprob=-0.0060  p=0.9940
        '<|im_end|>'  logprob=-0.0001  p=0.9999
     '<|endoftext|>'  logprob=-18.1875  p=0.0000
     '<|endoftext|>'  logprob=-14.6822  p=0.

In [ ]:
batch_size = 4
max_new_tokens = 32
temperature = 0.6  # <-- Pay close attention to this parameter: you will be asked to explore how changing it affects the results.
top_p = 0.9
sample_answers = 10  # we sample 10 responses for each question.

# Repeat each element in data list 10 times
data = [item for item in data_records for _ in range(sample_answers)]

#if the block takes too long to run, use a subset for initial inspection
#data = data[:100]

results = []
for i in tqdm(range(0, len(data), batch_size)):
    batch = data[i:i + batch_size]
    batch = [format_inputs(item) for item in batch]

    prompts_batch = [processor.apply_chat_template(item["messages"], add_generation_prompt=True) for item in batch]
    images_batch = [item["image"] for item in batch]

    inputs = processor(
        images=images_batch,
        text=prompts_batch,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False if temperature == 0 else True,
            temperature=temperature,
            top_p=top_p,
            return_dict_in_generate=True,
            output_scores=True,
            output_logits=True,
            pad_token_id=processor.tokenizer.pad_token_id,
        )

    input_ids = inputs["input_ids"]

    for b_idx, item in enumerate(batch):
        input_len = input_ids[b_idx].shape[0]

        generated_tokens = outputs.sequences[b_idx, input_len:]
        generated_text = processor.decode(generated_tokens, skip_special_tokens=True)

        logprobs = []
        for j, logits in enumerate(outputs.logits):  # extract the probability associated to each generated token
            if j >= generated_tokens.shape[0]:  # Avoid out-of-bounds if gen shorter than scores
                break
            token_id = outputs.sequences[b_idx, input_len + j].item()
            probs = torch.nn.functional.softmax(logits[b_idx], dim=-1)
            logprob = torch.log(probs[token_id]).item()
            token_str = processor.tokenizer.decode([token_id])
            logprobs.append((token_str, logprob))

        row = df[df["index"] == item["index"]].iloc[0]

        results.append({
            "index": item["index"],
            "question": item["question"],
            "gold_label": item["answer"],
            "category": item["category"],
            'clinical VQA task': item['clinical VQA task'],
            'department': item['department'],
            'perceptual granularity': item['perceptual granularity'],
            'modality': item['modality'],
            'original task': item['original task'],
            'image_path':item['image_path'],
            "options": row["options"],
            "prompt": prompts_batch[b_idx],
            "response": generated_text,
            "logprobs": logprobs
        })

# save the generated responses
output_df = pd.DataFrame(results)
output_folder = "outputs"
os.makedirs(output_folder, exist_ok=True)

filename = f"outputs_{os.path.split(model_id)[1]}-temp_{temperature}.tsv"
output_df.to_csv(os.path.join(output_folder, filename), index=False, sep="\t", encoding="utf-8")

## b. Inference with medgemma-4b-it

In [ ]:
import gc

# Release the previous model and tensors from its inference loop.
for name in (
    "model", "processor", "inputs", "outputs", "input_ids",
    "generated_tokens", "logits", "probs"
):
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor, GenerationConfig
from PIL import Image
import requests

model_id = "google/medgemma-4b-it"

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
model.device

In [ ]:
images_dir = "data/images"

def format_inputs(batch_item: Dict[str, Any]) -> Dict[str, Any]:
    image = Image.open(os.path.join(images_dir, batch_item["image_path"]))
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": batch_item["prompt"]},
                {"type": "image", "image":image},
            ],
        }
    ]

    return {
        **batch_item,
        "messages": messages,
        "image": [image],
    }

In [ ]:
batch_size = 1
max_new_tokens = 32
temperature = 0.6  # <-- Pay close attention to this parameter: you will be asked to explore how changing it affects the results.
top_p = 0.9
sample_answers = 10

# Repeat each element in data list 10 times
data = [item for item in data_records for _ in range(sample_answers)]

#if the block takes too long to run, use a subset for initial inspection
#data = data[:100]

results = []
for i in tqdm(range(0, len(data), batch_size)):
    batch = data[i:i + batch_size]
    batch = [format_inputs(item) for item in batch]

    prompts_batch = [processor.apply_chat_template(item["messages"], add_generation_prompt=True) for item in batch]
    images_batch = [item["image"] for item in batch]

    inputs = processor(
        text=prompts_batch,
        images=images_batch,
        padding=True,
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False if temperature == 0 else True,
            temperature=temperature,
            top_p=top_p,
            return_dict_in_generate=True,
            output_scores=True,
            output_logits=True,
            pad_token_id=processor.tokenizer.pad_token_id,
        )

    input_ids = inputs["input_ids"]

    for b_idx, item in enumerate(batch):
        input_len = input_ids[b_idx].shape[0]

        generated_tokens = outputs.sequences[b_idx, input_len:]
        generated_text = processor.decode(generated_tokens, skip_special_tokens=True)

        logprobs = []
        for j, logits in enumerate(outputs.scores):
            if j >= generated_tokens.shape[0]:
                break
            token_id = outputs.sequences[b_idx, input_len + j].item()
            probs = torch.nn.functional.softmax(logits[b_idx], dim=-1)
            logprob = torch.log(probs[token_id]).item()
            token_str = processor.tokenizer.decode([token_id])
            logprobs.append((token_str, logprob))

        row = df[df["index"] == item["index"]].iloc[0]

        results.append({
            "index": item["index"],
            "question": item["question"],
            "gold_label": item["answer"],
            "category": item["category"],
            'clinical VQA task': item['clinical VQA task'],
            'department': item['department'],
            'perceptual granularity': item['perceptual granularity'],
            'modality': item['modality'],
            'original task': item['original task'],
            'image_path': item['image_path'],
            "options": row["options"],
            "prompt": prompts_batch[b_idx],
            "response": generated_text,
            "logprobs": logprobs
        })

# save the generated responses
output_df = pd.DataFrame(results)
output_folder = "outputs"
os.makedirs(output_folder, exist_ok=True)

filename = f"outputs_{os.path.split(model_id)[1]}-temp_{temperature}.tsv"
output_df.to_csv(os.path.join(output_folder, filename), index=False, sep="\t", encoding="utf-8")

## Compute and Save Embeddings of Model Outputs
In Section 2, you will compute and evaluate several Uncertainty Estimation (UE) methods. One of these is Radial Dispersion Score (RDS), an embedding-based UE method. To prepare for this, we first need to compute and save the embeddings of all model responses.
We will:

1. Compute and save the embeddings for all model responses.
2. Zip the folder containing the embeddings.
3. Download the zip file.
4. Change the runtime to CPU (see Section 2) and upload the zip file to the runtime.

In [ ]:
# Load the embedding model
from sentence_transformers import SentenceTransformer
import numpy as np
import os

embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
os.makedirs("/content/data/embeddings", exist_ok=True)
embeddings_folder = "/content/data/embeddings"
outputs_folder = "/content/data/model_responses/"

def embed_save_outputs(df, output_path, embedding_model):
    sentences = df["response"].tolist()
    sentences = [str(s) for s in sentences]
    embeddings = embedding_model.encode(sentences, batch_size=32, show_progress_bar=True)
    np.save(output_path, embeddings)

for path in glob.glob(os.path.join(outputs_folder, "*")):
    filename = os.path.basename(path)
    filename = filename.replace("outputs_", "").replace(".tsv", "")
    embedding_file = os.path.join(embeddings_folder, f"embeddings_{filename}.npy")
    df = pd.read_csv(path, sep='\t')
    if not os.path.exists(embedding_file):
        embed_save_outputs(df, embedding_file, embedding_model)

In [ ]:
# We will zip the /content/data/embeddings folder, download the zip file manually, since restarting the runtime will delete the results.
! zip -r /content/data/embeddings.zip /content/data/embeddings

# 2. Load and Analyze Outputs (run on CPUs) - Uncertainty Quantification


In [1]:
import re
from collections import Counter
import ast
import numpy as np
import pandas as pd

### a. Utilities and Implementations for
* Answer extraction, accuracy, majority voting
* Uncertainty Estimation (UE) methods:
  * **Token-level probability** (log-likelihood): the probability assigned to each generated token in a response, used as a proxy for model uncertainty. Higher probability = higher confidence. Note: this is not available for `GPT-5-mini`. You have to implement this method (see the assignment report).
  * **Semantic Entropy**: measures the consistency of multiple responses to the same question. More consistent responses across runs = higher model confidence.
  * **Radial Dispersion Score (RDS)**: measures the dispersion of embeddings from their centroid, for multiple responses to the same question. You have to implement this method (see the assignment report). Note: Don't forget to upload the zip file with the saved embeddings, and unzip it!
* Uncertainty Estimation evaluation metrics:
  * Discrimination metric: Area Under the Receiver Operating Characteristic Curve (**AUROC**)
  * Calibration metrics: Expected Calibration Error (**ECE**) and **Brier Score**.


In [2]:
def parse_response(response):
    """Extract the letter choice from the response."""
    # Look for pattern like [A], [B], etc.
    match = re.search(r'\[([A-E])\]', response)
    if match:
        return match.group(1)
    return None

def parse_options(options_str):
    """Parse the options string to get the number of options."""
    try:
        # Convert string representation of dict to actual dict
        options_dict = ast.literal_eval(options_str)
        return len(options_dict)
    except:
        # Fallback: assume 5 options if parsing fails
        return 5

def load_and_process_data(file_path):
    """Load the TSV file and process the data."""
    df = pd.read_csv(file_path, sep='\t')

    # Parse responses to extract letter choices
    df['parsed_response'] = df['response'].apply(parse_response)

    # Parse options to get number of choices per question
    df['num_options'] = df['options'].apply(parse_options)

    return df

def compute_accuracy_stats(df):
    """Compute overall accuracy and per-question statistics."""
    stats = {}

    # Overall accuracy
    correct_predictions = (df['parsed_response'] == df['gold_label']).sum()
    total_predictions = len(df)
    overall_accuracy = correct_predictions / total_predictions

    stats['overall_accuracy'] = overall_accuracy
    stats['total_samples'] = total_predictions
    stats['correct_predictions'] = correct_predictions

    # Per-question statistics (grouped by index, which is unique per question)
    question_stats = []
    unique_indices = df['index'].unique()

    for idx in unique_indices:
        question_data = df[df['index'] == idx]

        # Get all responses for this question (should be 10 samples)
        responses = question_data['parsed_response'].tolist()
        gold_label = question_data['gold_label'].iloc[0]  # Should be same for all samples
        question_text = question_data['question'].iloc[0]  # Should be same for all samples
        num_options = question_data['num_options'].iloc[0]  # Should be same for all samples

        # Calculate accuracy for this question
        correct = sum(1 for r in responses if r == gold_label)
        accuracy = correct / len(responses)

        # Calculate consistency (how often the model gives the same answer)
        response_counts = Counter(responses)
        most_common_response, most_common_count = response_counts.most_common(1)[0]
        consistency = most_common_count / len(responses)

        # Calculate entropy of responses (lower = more consistent)
        response_probs = np.array(list(response_counts.values())) / len(responses)
        entropy = -np.sum(response_probs * np.log2(response_probs))

        question_stats.append({
            'index': idx,
            'question': question_text[:50] + '...' if len(question_text) > 50 else question_text,
            'gold_label': gold_label,
            'accuracy': accuracy,
            'consistency': consistency,
            'most_common_response': most_common_response,
            'entropy': entropy,
            'num_samples': len(responses),
            'num_options': num_options
        })

    stats['question_stats'] = question_stats
    return stats


In [3]:
def compute_semantic_entropy_uncertainty(df):
    """
    Compute semantic entropy as uncertainty metric and evaluate it.
    For each question, semantic entropy measures how uncertain the model is.
    """
    grouped = df.groupby('index')

    uncertainties = []
    correctness = []  # Binary: 1 if majority answer is correct, 0 otherwise
    num_options_list = []  # Track number of options for each question

    for idx, group in grouped:
        responses = group['parsed_response'].tolist()
        gold_label = group['gold_label'].iloc[0]
        num_options = group['num_options'].iloc[0]

        # Count responses
        response_counts = Counter(responses)
        total_responses = len(responses)

        # Calculate semantic entropy (uncertainty)
        response_probs = np.array(list(response_counts.values())) / total_responses
        semantic_entropy = -np.sum(response_probs * np.log2(response_probs))

        # Determine if majority answer is correct
        most_common_response = response_counts.most_common(1)[0][0]
        is_correct = 1 if most_common_response == gold_label else 0

        uncertainties.append(semantic_entropy)
        correctness.append(is_correct)
        num_options_list.append(num_options)

    return np.array(uncertainties), np.array(correctness), np.array(num_options_list)


def compute_calibration_metrics(uncertainties, correctness, n_bins=10):
    """Compute AUROC, ECE, and Brier score with dynamic normalization."""

    from sklearn.metrics import roc_auc_score

    # AUROC: How well does uncertainty predict incorrectness?
    # Higher uncertainty should predict lower correctness
    try:
        auroc = roc_auc_score(1 - correctness,
                              uncertainties)  # 1-correctness because higher uncertainty should predict incorrectness
    except ValueError:
        auroc = 0.5  # If all samples have same correctness

    # Expected Calibration Error (ECE)
    def compute_ece(uncertainties, correctness, n_bins=10):
        bin_edges = np.linspace(0, np.max(uncertainties) + 1e-8, n_bins + 1)
        ece = 0

        for i in range(n_bins):
            mask = (uncertainties >= bin_edges[i]) & (uncertainties < bin_edges[i + 1])
            if np.sum(mask) > 0:
                bin_accuracy = np.mean(correctness[mask])
                bin_confidence = 1 - np.mean(uncertainties[mask])
                bin_confidence = np.clip(bin_confidence, 0, 1)  # Ensure it's between 0 and 1
                bin_size = np.sum(mask) / len(uncertainties)
                ece += bin_size * abs(bin_accuracy - bin_confidence)

        return ece

    u_min, u_max = uncertainties.min(), uncertainties.max()
    if u_max > u_min:
        uncertainties = (uncertainties - u_min) / (u_max - u_min)
    else:
        uncertainties = np.zeros_like(uncertainties)
    confidences = 1 - uncertainties
    confidences = np.clip(confidences, 0, 1)

    ece = compute_ece(uncertainties, correctness, n_bins)
    brier_score = np.mean((confidences - correctness) ** 2)

    return auroc, ece, brier_score

### b. Run Evaluation

In [9]:
# Replace this with the output file corresponding to the model and decoding configuration you want to evaluate.
file_path = "./data/model_responses/outputs_Qwen2-VL-2B-Instruct-temp_1.0.tsv"

print("Loading and processing data...")
df = load_and_process_data(file_path)

print(f"Loaded {len(df)} samples")
print(f"Number of unique questions (indices): {df['index'].nunique()}")
print(f"Samples per question: {len(df) // df['index'].nunique()}")

# Print option distribution
option_counts = df.groupby('index')['num_options'].first().value_counts().sort_index()
print(f"Distribution of number of options:")
for num_opts, count in option_counts.items():
    print(f"  {num_opts} options: {count} questions")

Loading and processing data...
Loaded 3000 samples
Number of unique questions (indices): 300
Samples per question: 10
Distribution of number of options:
  2 options: 7 questions
  3 options: 1 questions
  4 options: 210 questions
  5 options: 82 questions


In [ ]:
# Basic accuracy statistics
print("\n" + "=" * 50)
print("ACCURACY ANALYSIS")
print("=" * 50)

stats = compute_accuracy_stats(df)
print(f"Overall Accuracy: {stats['overall_accuracy']:.3f}")
print(f"Correct Predictions: {stats['correct_predictions']}/{stats['total_samples']}")

# Per-question statistics
print(f"\nPer-Question Statistics:")
print("-" * 100)
print(f"{'Index':<6} {'Question':<50} {'Gold':<5} {'Opts':<7} {'Acc':<6} {'Cons':<2} {'Pred':<4} {'Entropy':<7}")
print("-" * 100)
for q_stat in stats['question_stats']:
    print(f"{q_stat['index']:<6} {q_stat['question']:<40} {q_stat['gold_label']:<4} "
          f"{q_stat['num_options']:<4} {q_stat['accuracy']:<6.3f} {q_stat['consistency']:<6.3f} "
          f"{q_stat['most_common_response']:<4} {q_stat['entropy']:<7.3f}")

In [ ]:
# Uncertainty quantification analysis
print("\n" + "=" * 60)
print("UNCERTAINTY QUANTIFICATION ANALYSIS - Semantic Entropy")
print("=" * 60)

uncertainties, correctness, num_options_array = compute_semantic_entropy_uncertainty(df)
auroc, ece, brier_score = compute_calibration_metrics(uncertainties, correctness)

print(f"\n Discrimination and Calibration Metrics:\n")
print(f" * AUROC (uncertainty vs incorrectness): {auroc:.3f}")
print(f" * Expected Calibration Error (ECE): {ece:.3f}")
print(f" * Brier Score: {brier_score:.3f}")

In [10]:
q5 = load_and_process_data("./data/model_responses/outputs_Qwen2-VL-2B-Instruct-temp_1.0.tsv")  # own variable, df untouched
qid = 398
q = q5[q5["index"] == qid]
print(f"gold: {q['gold_label'].iloc[0]}  options: {q['options'].iloc[0]}\n")
for resp, letter in zip(q["response"], q["parsed_response"]):
    print(f"  {letter!s:>4}  {resp!r}")

counts = Counter(q["parsed_response"])
print("\ncounts:", dict(counts))
manual = 0
for letter, c in counts.items():
    p = c / len(q)
    manual -= p * np.log2(p)
    print(f"  {letter}: p={p:.1f}, -p*log2(p)={-p * np.log2(p):.4f}")

se, _, _ = compute_semantic_entropy_uncertainty(q5)
code_se = dict(zip(sorted(q5["index"].unique()), se))[qid]  # groupby iterates indices in sorted order
print(f"\nmanual entropy: {manual:.4f} bits, code entropy: {code_se:.4f} bits")

n_none_q = q5.groupby("index")["parsed_response"].apply(lambda s: s.isna().any()).sum()
print(f"\nunparseable (None) responses in file: {q5['parsed_response'].isna().sum()}/{len(q5)}, "
      f"questions with at least one None: {n_none_q}/{q5['index'].nunique()}")

gold: D  options: {'A': 'fascia uncovered kidney parenchyma', 'B': 'lung', 'C': 'fascia covered kidney parenchyma', 'D': 'heart', 'E': 'clavicles'}

     B  '[B] lung'
     C  '[C] fascia covered kidney parenchyma'
     C  '[C] fascia covered kidney parenchyma'
     B  '[B] lung'
     C  '[C] fascia covered kidney parenchyma'
     B  '[B] lung'
     B  '[B] lung'
     B  '[B] lung'
     D  '[D] heart'
     B  '[B] lung'

counts: {'B': 6, 'C': 3, 'D': 1}
  B: p=0.6, -p*log2(p)=0.4422
  C: p=0.3, -p*log2(p)=0.5211
  D: p=0.1, -p*log2(p)=0.3322

manual entropy: 1.2955 bits, code entropy: 1.2955 bits

unparseable (None) responses in file: 21/3000, questions with at least one None: 18/300


In [12]:
rows, summary, dfs = [], [], {}
for t in ["0.2", "0.6", "1.0"]:
    d = load_and_process_data(f"./data/model_responses/outputs_Qwen2.5-VL-32B-Instruct-temp_{t}.tsv")
    dfs[t] = d
    g = d.groupby("index")
    print(f"T={t}: samples/question {g.size().unique().tolist()}, gold constant {(g['gold_label'].nunique() == 1).all()}, "
          f"options constant {(g['options'].nunique() == 1).all()}, shuffle_index {sorted(d['shuffle_index'].unique().tolist())}")

    _, correctness_t, _ = compute_semantic_entropy_uncertainty(d)  # notebook majority rule
    for (idx, grp), maj_ok in zip(g, correctness_t):
        ranked = Counter(grp["parsed_response"]).most_common()
        tied = [k for k, v in ranked if v == ranked[0][1]]
        gold = grp["gold_label"].iloc[0]
        n_ok = (grp["parsed_response"] == gold).sum()
        rows.append({"index": idx, "temperature": float(t), "gold": gold, "n_correct": n_ok,
                     "frac_correct": n_ok / len(grp), "majority": ranked[0][0], "majority_correct": bool(maj_ok),
                     "tie": len(tied) > 1, "gold_in_tie": len(tied) > 1 and gold in tied,
                     "n_none": grp["parsed_response"].isna().sum()})

    r = pd.DataFrame(rows)
    r = r[r["temperature"] == float(t)]
    summary.append({
        "T": t,
        "per_sample_acc": (d["parsed_response"] == d["gold_label"]).mean(),  # same as compute_accuracy_stats
        "majority_acc": r["majority_correct"].mean(),
        "maj_acc_gold_ties_wrong": (r["majority_correct"] & ~r["gold_in_tie"]).mean(),
        "maj_acc_gold_ties_right": (r["majority_correct"] | r["gold_in_tie"]).mean(),
        "none_responses": int(d["parsed_response"].isna().sum()),
        "majority_is_none": int(r["majority"].isna().sum()),
        "tied_questions": int(r["tie"].sum()),
        "ties_with_gold": int(r["gold_in_tie"].sum()),
        "type1_maj_wrong_frac>=0.4": int((~r["majority_correct"] & (r["frac_correct"] >= 0.4)).sum()),
        "type2_maj_right_frac<0.5": int((r["majority_correct"] & (r["frac_correct"] < 0.5)).sum()),
        "type3_0<frac<1": int(((r["frac_correct"] > 0) & (r["frac_correct"] < 1)).sum()),
    })

q8 = pd.DataFrame(rows)
print("\n" + pd.DataFrame(summary).set_index("T").T.round(4).to_string())

# example at T=1.0: first question by index of the most informative available type
r10 = q8[q8["temperature"] == 1.0].sort_values("index")
for name, mask in [("type1", ~r10["majority_correct"] & (r10["frac_correct"] >= 0.4)),
                   ("type2", r10["majority_correct"] & (r10["frac_correct"] < 0.5)),
                   ("type3", (r10["frac_correct"] > 0) & (r10["frac_correct"] < 1))]:
    if mask.any():
        ex = r10[mask].iloc[0]
        break
grp = dfs["1.0"][dfs["1.0"]["index"] == ex["index"]]
print(f"\nexample ({name}) at T=1.0, index {ex['index']}: gold {ex['gold']}, frac_correct {ex['frac_correct']:.1f}, "
      f"majority {ex['majority']}, majority_correct {ex['majority_correct']}, tie {ex['tie']}")
print(f"question: {grp['question'].iloc[0]}\noptions: {grp['options'].iloc[0]}")
print("votes:", dict(Counter(grp["parsed_response"]).most_common()))
for resp, letter in zip(grp["response"], grp["parsed_response"]):
    print(f"  {letter!s:>4}  {resp!r}")

os.makedirs("outputs", exist_ok=True)
q8.to_csv("outputs/q8_qwen2.5-vl-32b_majority_vote_per_question.csv", index=False)

T=0.2: samples/question [10], gold constant True, options constant True, shuffle_index [0]
T=0.6: samples/question [10], gold constant True, options constant True, shuffle_index [0]
T=1.0: samples/question [10], gold constant True, options constant True, shuffle_index [0]

T                              0.2      0.6       1.0
per_sample_acc              0.5140   0.4973    0.4963
majority_acc                0.5233   0.5100    0.5167
maj_acc_gold_ties_wrong     0.5100   0.5000    0.5033
maj_acc_gold_ties_right     0.5233   0.5300    0.5300
none_responses              0.0000   8.0000   20.0000
majority_is_none            0.0000   0.0000    0.0000
tied_questions              6.0000  13.0000   13.0000
ties_with_gold              4.0000   9.0000    8.0000
type1_maj_wrong_frac>=0.4   0.0000  10.0000   12.0000
type2_maj_right_frac<0.5    0.0000   1.0000    1.0000
type3_0<frac<1             23.0000  77.0000  136.0000

example (type1) at T=1.0, index 834: gold B, frac_correct 0.4, majority C, ma